# Regresión polinomial en Python

Este notebook presenta un ejemplo completo y reproducible de **regresión polinomial** utilizando un CSV incluido en el proyecto.

El objetivo es predecir `ingresos` a partir de `edad`, `horas_estudio` y `experiencia`.

La regresión polinomial permite modelar relaciones no lineales mediante la creación de potencias e interacciones de las variables originales.

Se calculan **R², RMSE, MAE, Pearson y Bias**.

## 1. Librerías

Se utiliza `PolynomialFeatures` para transformar las variables originales y `LinearRegression` para ajustar el modelo sobre las características polinómicas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

pd.set_option('display.precision', 4)

## 2. Carga del CSV

El fichero `datos_regresion_polinomial.csv` está incluido en la misma carpeta que este notebook.

In [ ]:
df = pd.read_csv('datos_regresion_polinomial.csv')

print(f'Dimensiones del dataset: {df.shape}')
display(df.head())

## 3. Exploración inicial

Antes de entrenar el modelo comprobamos la estructura, los valores ausentes y los principales estadísticos descriptivos.

In [ ]:
print('Información del dataset:')
df.info()

print('\nValores ausentes:')
display(df.isnull().sum().to_frame('missing_values'))

print('\nEstadísticas descriptivas:')
display(df.describe())

## 4. Matriz de correlación

La matriz de correlación permite observar las relaciones lineales entre las variables. Es importante recordar que Pearson puede no detectar adecuadamente relaciones puramente no lineales.

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(
    df.drop(columns='id').corr(),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0
)
plt.title('Matriz de correlación')
plt.tight_layout()
plt.show()

## 5. Variables predictoras y objetivo

La regresión polinomial de una variable puede expresarse como:

$$
\hat{y} = \beta_0 + \beta_1x + \beta_2x^2 + \cdots + \beta_dx^d
$$

Con varias variables, `PolynomialFeatures` genera también términos de interacción.

En este ejemplo:

- Variables predictoras: `edad`, `horas_estudio`, `experiencia`.
- Variable objetivo: `ingresos`.

In [ ]:
features = ['edad', 'horas_estudio', 'experiencia']
target = 'ingresos'

X = df[features]
y = df[target]

print('Variables predictoras:', features)
print('Variable objetivo:', target)

## 6. División entrenamiento/prueba

Se utiliza el 80 % de las observaciones para entrenamiento y el 20 % para evaluación.

El `random_state=42` permite reproducir la misma partición.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(f'Tamaño entrenamiento: {len(X_train)}')
print(f'Tamaño prueba: {len(X_test)}')

## 7. Construcción del modelo polinomial

Se utiliza un `Pipeline` con dos etapas:

1. `PolynomialFeatures`: genera las características polinómicas.
2. `LinearRegression`: ajusta una regresión lineal sobre esas características.

Aunque el estimador final es lineal respecto a los coeficientes, el modelo representa una relación polinómica respecto a las variables originales.

In [ ]:
degree = 2

model = Pipeline([
    ('poly', PolynomialFeatures(
        degree=degree,
        include_bias=False
    )),
    ('linear', LinearRegression())
])

model.fit(X_train, y_train)

print(f'Modelo polinomial de grado {degree} entrenado correctamente.')

## 8. Características generadas

La transformación polinomial aumenta el número de variables. Para grado 2 aparecen términos como `edad²`, `horas_estudio²`, `experiencia²` y las interacciones entre variables.

Esto aumenta la flexibilidad del modelo, pero grados demasiado elevados pueden provocar sobreajuste.

In [ ]:
poly = model.named_steps['poly']

feature_names = poly.get_feature_names_out(features)

print(f'Variables originales: {len(features)}')
print(f'Variables polinómicas: {len(feature_names)}')
print('\nCaracterísticas generadas:')

for name in feature_names:
    print('-', name)

## 9. Predicciones

Se calculan las predicciones sobre el conjunto de prueba, que no ha participado en el ajuste del modelo.

In [ ]:
y_pred = model.predict(X_test)

resultados = pd.DataFrame({
    'real': y_test.values,
    'predicho': y_pred,
    'error': y_test.values - y_pred
})

display(resultados)

## 10. Cálculo de métricas

Se utilizan cinco métricas.

### R²

Mide la proporción de variabilidad explicada por el modelo.

### RMSE

$$
RMSE =
\sqrt{
\frac{1}{n}
\sum_{i=1}^{n}
(y_i-\hat{y}_i)^2
}
$$

### MAE

$$
MAE =
\frac{1}{n}
\sum_{i=1}^{n}
|y_i-\hat{y}_i|
$$

### Pearson

Mide la correlación lineal entre los valores reales y predichos.

### Bias

En este notebook se define como:

$$
Bias =
\frac{1}{n}
\sum_{i=1}^{n}
(y_i-\hat{y}_i)
$$

Un Bias positivo indica una tendencia media a subestimar; un Bias negativo indica una tendencia media a sobreestimar.

In [ ]:
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
pearson, pearson_pvalue = pearsonr(y_test, y_pred)
bias = np.mean(y_test.to_numpy() - y_pred)

metricas = pd.DataFrame({
    'Metrica': ['R2', 'RMSE', 'MAE', 'Pearson', 'Bias'],
    'Valor': [r2, rmse, mae, pearson, bias]
})

display(metricas)

print(f'p-value de Pearson: {pearson_pvalue:.6f}')

## 11. Comparación de grados

Se comparan modelos de grados 1, 2 y 3.

El grado 1 equivale a una regresión lineal. Los grados superiores permiten capturar relaciones más complejas.

En aplicaciones reales, la selección del grado debería realizarse preferentemente mediante validación cruzada.

In [ ]:
resultados_grados = []

for degree_test in [1, 2, 3]:
    m = Pipeline([
        ('poly', PolynomialFeatures(
            degree=degree_test,
            include_bias=False
        )),
        ('linear', LinearRegression())
    ])

    m.fit(X_train, y_train)
    pred = m.predict(X_test)

    p, _ = pearsonr(y_test, pred)

    resultados_grados.append({
        'grado': degree_test,
        'R2': r2_score(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'MAE': mean_absolute_error(y_test, pred),
        'Pearson': p,
        'Bias': np.mean(y_test.to_numpy() - pred)
    })

comparacion = pd.DataFrame(resultados_grados)
display(comparacion)

## 12. Valores reales frente a predicciones

La diagonal representa la situación ideal en la que:

$$
y = \hat{y}
$$

Cuanto más próximos estén los puntos a esta diagonal, menores serán los errores de predicción.

In [ ]:
plt.figure(figsize=(7, 6))

plt.scatter(y_test, y_pred, s=70)

lo = min(y_test.min(), y_pred.min())
hi = max(y_test.max(), y_pred.max())

plt.plot(
    [lo, hi],
    [lo, hi],
    linestyle='--'
)

plt.xlabel('Valores reales')
plt.ylabel('Valores predichos')
plt.title('Regresión polinomial: real vs. predicho')
plt.tight_layout()
plt.show()

## 13. Análisis de residuos

El residuo de cada observación se define como:

$$
e_i = y_i - \hat{y}_i
$$

El análisis de residuos permite detectar patrones sistemáticos que pueden indicar problemas en el modelo.

In [ ]:
residuos = y_test.to_numpy() - y_pred

plt.figure(figsize=(8, 5))
sns.histplot(residuos, kde=True)
plt.axvline(0, linestyle='--')
plt.xlabel('Residuo (real - predicho)')
plt.title('Distribución de los residuos')
plt.tight_layout()
plt.show()

## 14. Residuos frente a predicciones

Una segunda visualización permite comprobar si los residuos presentan algún patrón relacionado con el nivel de predicción.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuos, s=70)
plt.axhline(0, linestyle='--')
plt.xlabel('Valores predichos')
plt.ylabel('Residuos')
plt.title('Residuos frente a valores predichos')
plt.tight_layout()
plt.show()

## 15. Resumen final

El flujo completo es:

**CSV → exploración → selección de variables → train/test split → transformación polinomial → entrenamiento → predicción → métricas → comparación de grados → análisis de residuos.**

La regresión polinomial puede capturar relaciones no lineales, pero aumentar el grado incrementa la complejidad y puede producir sobreajuste.

In [ ]:
print('===== RESULTADOS REGRESIÓN POLINOMIAL =====')
print(f'Grado   : {degree}')
print(f'R²      : {r2:.4f}')
print(f'RMSE    : {rmse:.4f}')
print(f'MAE     : {mae:.4f}')
print(f'Pearson : {pearson:.4f}')
print(f'Bias    : {bias:.4f}')